In [1]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")

Project root: /Users/matthewho/Documents/research/ctx_editor


In [2]:
# imported log folder at
# data/imported_lic_logs/sota_model_logs
# LIC_LOGS_DIR = DATA_DIR / "imported_lic_logs/sota_model_logs"
LIC_LOGS_DIR = PROJECT_ROOT / "src/lic/logs"

# Load all JSONL log files into a single list of records
all_records = []
for jsonl_file in sorted(LIC_LOGS_DIR.rglob("*.jsonl")):
    records = read_jsonl(jsonl_file)
    all_records.extend(records)
    print(f"  {jsonl_file.relative_to(LIC_LOGS_DIR)}: {len(records)} records")

print(f"\nTotal records loaded: {len(all_records)}")

  aime/full/full_aime_gpt-5-mini.jsonl: 31 records
  aime/sharded/sharded_aime_gpt-5-mini.jsonl: 31 records
  code/full/full_code_gpt-5-mini.jsonl: 30 records
  code/sharded/sharded_code_gpt-5-mini.jsonl: 30 records

Total records loaded: 122


In [3]:
# ── Step 1: Establish canonical correctness for each record ──
# Reconcile is_correct (bool|None) and score (float|None/NaN) into a single canonical value.

both_null_conv_ids = []
mismatch_conv_ids = []

def get_canonical(is_correct, score):
    """Return canonical correctness (True/False/None) and flag category."""
    ic_null = is_correct is None
    sc_null = score is None or (isinstance(score, float) and np.isnan(score))

    if ic_null and sc_null:
        return None, "both_null"
    elif (not ic_null) and (not sc_null):
        # Both present — check agreement
        if is_correct != (score == 1.0):
            return is_correct, "mismatch"
        return is_correct, "ok"
    else:
        # One is non-null — derive from whichever is available
        if not ic_null:
            return is_correct, "ok"
        else:
            return (score == 1.0), "ok"

for r in all_records:
    canonical, flag = get_canonical(r.get("is_correct"), r.get("score"))
    r["canonical_correct"] = canonical
    if flag == "both_null":
        both_null_conv_ids.append(r["conv_id"])
    elif flag == "mismatch":
        mismatch_conv_ids.append(r["conv_id"])

print(f"Total records: {len(all_records)}")
print(f"  Both null (is_correct=None, score=None/NaN): {len(both_null_conv_ids)}")
print(f"  Mismatch (both non-null but disagree):       {len(mismatch_conv_ids)}")
print(f"  Consistent / single-source:                  {len(all_records) - len(both_null_conv_ids) - len(mismatch_conv_ids)}")

if mismatch_conv_ids:
    print(f"\n⚠ Mismatched conv_ids (both non-null but disagree):")
    mismatch_records = [r for r in all_records if r["conv_id"] in set(mismatch_conv_ids)]
    display(pd.DataFrame([{
        "conv_id": r["conv_id"], "task": r["task"], "task_id": r["task_id"],
        "model": r["assistant_model"], "conv_type": r["conv_type"],
        "is_correct": r.get("is_correct"), "score": r.get("score"),
    } for r in mismatch_records]))

Total records: 122
  Both null (is_correct=None, score=None/NaN): 0
  Mismatch (both non-null but disagree):       0
  Consistent / single-source:                  122


In [4]:
# ── Step 2: Timestamp-based deduplication ──
# Extract timestamp from last trace entry for each record.
# If [task, model, conv_type, task_id] has multiple entries, keep the latest one.

def get_last_timestamp(record):
    """Extract timestamp from last trace entry (answer-evaluation or conversation-completed)."""
    trace = record.get("trace", [])
    for entry in reversed(trace):
        if "timestamp" in entry:
            return entry["timestamp"]
    return None

# Build flat records with timestamp, using canonical_correct from Step 1
flat_records = []
for r in all_records:
    canonical = r["canonical_correct"]
    flat_records.append({
        "conv_id": r["conv_id"],
        "task": r["task"],
        "task_id": r["task_id"],
        "model": r["assistant_model"],
        "conv_type": r["conv_type"],
        "canonical_correct": canonical,
        # Numeric score: True→1.0, False→0.0, None→NaN
        "score": 1.0 if canonical is True else (0.0 if canonical is False else np.nan),
        "timestamp": get_last_timestamp(r),
    })

df_all = pd.DataFrame(flat_records)
df_all["timestamp"] = pd.to_datetime(df_all["timestamp"])

print(f"Total records before dedup: {len(df_all)}")

# Count duplicates per key
dedup_key = ["task", "model", "conv_type", "task_id"]
dup_counts = df_all.groupby(dedup_key).size()
dups = dup_counts[dup_counts > 1]
print(f"Duplicate groups (same task/model/conv_type/task_id): {len(dups)}")
if len(dups) > 0:
    print(f"Max duplicates in a group: {dups.max()}")
    print(f"Total duplicate records to discard: {dups.sum() - len(dups)}")

# Keep the latest (most recent timestamp) for each group
df_all = df_all.sort_values("timestamp", ascending=False)
df_deduped = df_all.drop_duplicates(subset=dedup_key, keep="first").copy()
df_deduped = df_deduped.sort_values(dedup_key).reset_index(drop=True)

print(f"\nTotal records after dedup: {len(df_deduped)}")
print(f"  of which canonical_correct is None (NaN score): {df_deduped['score'].isna().sum()}")
print(f"\nRecords per (task, conv_type):")
display(df_deduped.groupby(["task", "conv_type"]).size().unstack(fill_value=0))

Total records before dedup: 122
Duplicate groups (same task/model/conv_type/task_id): 4
Max duplicates in a group: 2
Total duplicate records to discard: 4

Total records after dedup: 118
  of which canonical_correct is None (NaN score): 0

Records per (task, conv_type):


conv_type,full,sharded
task,,
aime,30,30
code,29,29


In [5]:
# ── Step 3: Aggregate summary DataFrame ──
# Columns: [task, model, conv_type, num_convs, total_score, average_score]

df_agg = (
    df_deduped
    .groupby(["task", "model", "conv_type"])
    .agg(
        num_convs=("score", "size"),
        total_score=("score", "sum"),
        average_score=("score", "mean"),
    )
    .reset_index()
    .sort_values(["task", "model", "conv_type"])
    .reset_index(drop=True)
)

print(f"Aggregate table: {len(df_agg)} rows")
display(df_agg)

Aggregate table: 4 rows


,task,model,conv_type,num_convs,total_score,average_score
0,aime,gpt-5-mini,full,30,13.0,0.433333
1,aime,gpt-5-mini,sharded,30,11.0,0.366667
2,code,gpt-5-mini,full,29,20.0,0.689655
3,code,gpt-5-mini,sharded,29,4.0,0.137931


In [7]:
print(df_agg.to_markdown())

|    | task   | model      | conv_type   |   num_convs |   total_score |   average_score |
|---:|:-------|:-----------|:------------|------------:|--------------:|----------------:|
|  0 | aime   | gpt-5-mini | full        |          30 |            13 |        0.433333 |
|  1 | aime   | gpt-5-mini | sharded     |          30 |            11 |        0.366667 |
|  2 | code   | gpt-5-mini | full        |          29 |            20 |        0.689655 |
|  3 | code   | gpt-5-mini | sharded     |          29 |             4 |        0.137931 |


In [9]:
# ── Step 4: Per-task_id pivot DataFrame ──
# Rows: unique task_id, Columns: {conv_type}_{model}
# Cells: 1.0 (correct), 0.0 (incorrect), or NaN (no entry)

# Create column name for each combination
df_deduped["col_name"] = df_deduped["conv_type"] + "_" + df_deduped["model"]

# Pivot: each row is a (task, task_id), columns are conv_type_model combos
df_pivot = df_deduped.pivot_table(
    index=["task", "task_id"],
    columns="col_name",
    values="score",
    aggfunc="first",  # should be unique after dedup
)

# Sort columns: group by conv_type then model
df_pivot = df_pivot.reindex(sorted(df_pivot.columns), axis=1)
df_pivot = df_pivot.reset_index()

print(f"Pivot table: {df_pivot.shape[0]} rows x {df_pivot.shape[1]} columns")
print(f"\nColumns: {list(df_pivot.columns)}")
print(f"\nNull counts per column:")
display(df_pivot.isnull().sum())
print()
display(df_pivot.head(20))

Pivot table: 415 rows x 16 columns

Columns: ['task', 'task_id', 'full_gpt-5', 'full_gpt-5-chat', 'full_gpt-5-mini', 'full_gpt-5.1', 'full_gpt-5.1-chat', 'full_gpt-5.2', 'full_gpt-5.2-chat', 'sharded_gpt-5', 'sharded_gpt-5-chat', 'sharded_gpt-5-mini', 'sharded_gpt-5.1', 'sharded_gpt-5.1-chat', 'sharded_gpt-5.2', 'sharded_gpt-5.2-chat']

Null counts per column:


col_name
task                     0
task_id                  0
full_gpt-5               0
full_gpt-5-chat          0
full_gpt-5-mini          0
full_gpt-5.1             0
full_gpt-5.1-chat        0
full_gpt-5.2             0
full_gpt-5.2-chat        0
sharded_gpt-5           64
sharded_gpt-5-chat       4
sharded_gpt-5-mini      11
sharded_gpt-5.1          8
sharded_gpt-5.1-chat     9
sharded_gpt-5.2          4
sharded_gpt-5.2-chat     7
dtype: int64

col_name,task,task_id,full_gpt-5,full_gpt-5-chat,full_gpt-5-mini,full_gpt-5.1,full_gpt-5.1-chat,full_gpt-5.2,full_gpt-5.2-chat,sharded_gpt-5,sharded_gpt-5-chat,sharded_gpt-5-mini,sharded_gpt-5.1,sharded_gpt-5.1-chat,sharded_gpt-5.2,sharded_gpt-5.2-chat
0,actions,sharded-BFCL/parallel_0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,actions,sharded-BFCL/parallel_1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,actions,sharded-BFCL/parallel_102,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
3,actions,sharded-BFCL/parallel_103,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,actions,sharded-BFCL/parallel_105,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5,actions,sharded-BFCL/parallel_107,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,actions,sharded-BFCL/parallel_109,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,actions,sharded-BFCL/parallel_111,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
8,actions,sharded-BFCL/parallel_113,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
9,actions,sharded-BFCL/parallel_116,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
